# Week 4 Product Analytics

This notebook answers one practical question: which products make money, and which ones are sitting on the shelf doing little or nothing? We will review product revenue, profit, speed of sale, slow-moving stock, and monthly demand patterns using the cleaned sales and stock data.

Key rules used:
- Use product-level revenue, units, and profit per product and per category.
- Define fast-moving products as the top quartile by units sold.
- Define slow-moving products as the bottom quartile by units sold, with stock value tied up highlighted.
- Dead stock is defined here as no OUT movement in more than 365 days; in this dataset no product meets that rule.
- Do not use running_balance for stock timing decisions. Use current_stock and movement flows instead.

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

root = Path(r'c:/Users/tayod/OneDrive/Desktop/heavy-supplier-warehouse-analytics')
sales = pd.read_csv(root / 'data' / 'processed' / 'sales_table.csv', parse_dates=['order_date', 'delivery_date', 'invoice_date', 'due_date', 'last_payment_date'])
products = pd.read_csv(root / 'data' / 'cleaned' / 'products.csv', parse_dates=['last_purchase_date'])
inventory = pd.read_csv(root / 'data' / 'cleaned' / 'inventory_master.csv')
stock = pd.read_csv(root / 'data' / 'processed' / 'purchases_fact_table.csv', parse_dates=['movement_date'])

print('Sales rows:', len(sales))
print('Product rows:', len(products))
print('Inventory rows:', len(inventory))
print('Movement rows:', len(stock))


In [ ]:
product_meta = products[['product_id', 'product_name', 'category', 'unit_cost']].drop_duplicates()
sales_merged = sales.merge(product_meta, on='product_id', how='left')
sales_merged['effective_unit_price'] = sales_merged['line_total'] / sales_merged['quantity']
sales_merged['profit_per_line'] = (sales_merged['effective_unit_price'] - sales_merged['unit_cost']) * sales_merged['quantity']

product_perf = (
    sales_merged.groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(revenue=('line_grand_total', 'sum'), units=('quantity', 'sum'), profit=('profit_per_line', 'sum'))
)
product_perf['margin_pct'] = product_perf['profit'] / product_perf['revenue']
product_perf = product_perf.merge(inventory.groupby('product_id', as_index=False)['current_stock'].sum(), on='product_id', how='left')
product_perf = product_perf.merge(product_meta[['product_id', 'unit_cost']], on='product_id', how='left')
product_perf['stock_value'] = product_perf['current_stock'].fillna(0) * product_perf['unit_cost']

product_top = product_perf.sort_values('revenue', ascending=False).head(10)
display(product_top[['product_name', 'category', 'revenue', 'units', 'profit', 'current_stock', 'stock_value']])

category_perf = product_perf.groupby('category', as_index=False).agg(revenue=('revenue', 'sum'), units=('units', 'sum'), profit=('profit', 'sum'))
category_perf['margin_pct'] = category_perf['profit'] / category_perf['revenue']
display(category_perf.sort_values('revenue', ascending=False))


### What this means

The high-revenue products are the ones delivering the biggest contributions to sales and profit. In a shop, these are the items that pay the rent and keep the business healthy. The category view shows which product groups are carrying the volume and which ones are weaker performers.

A product can sell well but still carry a heavy stock balance. That is why we must also review stock value alongside revenue and profit.

In [ ]:
q75 = product_perf['units'].quantile(0.75)
fast_moving = product_perf[product_perf['units'] >= q75].sort_values('units', ascending=False).head(10)
print('Fast-moving rule: top quartile by units sold; q75 =', q75)
display(fast_moving[['product_name', 'category', 'units', 'revenue', 'profit', 'current_stock', 'stock_value']])

plt.figure(figsize=(10, 6))
plt.bar(fast_moving['product_name'], fast_moving['units'])
plt.title('Fast-Moving Products (Top Quartile by Units Sold)')
plt.xlabel('Product')
plt.ylabel('Units Sold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


### What this means

Fast-moving products are the ones you should keep fully stocked. They are selling consistently and drive demand, so they are the most important products to watch for stockouts and replenishment timing.

In [ ]:
q25 = product_perf['units'].quantile(0.25)
slow_moving = product_perf[product_perf['units'] <= q25].sort_values('stock_value', ascending=False).head(10)
out_dates = stock[stock['movement_type'] == 'OUT'].groupby('product_id', as_index=False)['movement_date'].max()
out_dates.columns = ['product_id', 'last_out_date']
product_perf_with_dates = product_perf.merge(out_dates, on='product_id', how='left')
end_date = max(sales_merged['order_date'].max(), stock['movement_date'].max())
product_perf_with_dates['days_since_last_out'] = (end_date - product_perf_with_dates['last_out_date']).dt.days
dead_stock = product_perf_with_dates[(product_perf_with_dates['last_out_date'].isna()) | (product_perf_with_dates['days_since_last_out'] > 365)].sort_values('stock_value', ascending=False)

print('Slow-moving rule: bottom quartile by units sold; q25 =', q25)
display(slow_moving[['product_name', 'category', 'units', 'current_stock', 'stock_value']])
print('Dead-stock rule: no OUT movement for more than 365 days or no OUT movement at all.')
display(dead_stock[['product_name', 'category', 'current_stock', 'stock_value', 'days_since_last_out']])


### What this means

There are no true dead-stock items under the strict rule used here, which is useful information in itself. It means the business is still moving product, but some items are likely overstocked because they sit with large inventory balances even when not the highest-selling products.

In [ ]:
monthly_demand = sales_merged.assign(month=sales_merged['order_date'].dt.to_period('M').astype(str))
monthly_product = monthly_demand.groupby(['month', 'product_id', 'product_name', 'category'], as_index=False)['quantity'].sum()
monthly_category = monthly_demand.groupby(['month', 'category'], as_index=False)['quantity'].sum()

print('Monthly product demand sample:')
display(monthly_product.sort_values(['month', 'quantity'], ascending=[True, False]).head(10))

plt.figure(figsize=(12, 7))
for category_name, group in monthly_category.sort_values('month').groupby('category'):
    plt.plot(group['month'], group['quantity'], marker='o', label=category_name)
plt.title('Monthly Demand by Category')
plt.xlabel('Month')
plt.ylabel('Units Sold')
plt.xticks(rotation=45)
plt.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


### Final takeaways

This Week 4 view tells the story of which products are truly making money, which products are moving quickly, and which items are tying up stock value without matching the same sales performance. The early signal from the project is clear: the stock value is much larger than revenue, and the product analysis is the first place where that overstock pattern becomes visible.